# 01. Setup and Corpus Preparation

**Goal**: end this notebook with a populated Qdrant collection (`eval_corpus`)
and a known list of source documents we can author queries against.

Steps:
1. Verify Qdrant is reachable.
2. Confirm the corpus files are present under `data/corpus/`.
3. Run a programmatic ingest (mirrors `agent-kms ingest --reset`).
4. Print corpus statistics (chunks / docs / heading distribution).

The corpus itself is **already committed** to `data/corpus/`. It consists of
15 markdown files drawn from two public CC-BY-licensed sources — see
`data/corpus/LICENSES.md` for attribution.


In [1]:
import os
import sys
from pathlib import Path

# Resolve project root so the notebook works whether the CWD is the
# notebooks/ subdir (default for Jupyter) or the project root.
ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

# Isolate from personal agent-kms data: dedicated Qdrant instance (port 6433)
# AND a dedicated collection name. The personal Qdrant on :6333 is untouched.
os.environ["QDRANT_URL"] = "http://localhost:6433"
os.environ["AGENT_KMS_COLLECTION"] = "eval_corpus"
os.environ["AGENT_KMS_CONFIG"] = str(ROOT / ".agent-kms" / "kms.toml")
os.environ["AGENT_KMS_KNOWLEDGE_ROOT"] = str(ROOT / "data" / "corpus")

sys.path.insert(0, str(ROOT))
print("project root:", ROOT)
print("qdrant:      ", os.environ["QDRANT_URL"])
print("collection:  ", os.environ["AGENT_KMS_COLLECTION"])


project root: /Users/toshihiro-yamaguchi/dev/portfolio/rag-evaluation-jp
qdrant:       http://localhost:6433
collection:   eval_corpus


## 1. Qdrant reachability

In [2]:
from agent_kms.store import get_client

client = get_client()
existing = [c.name for c in client.get_collections().collections]
print("collections on this Qdrant:", existing)


/Users/toshihiro-yamaguchi/dev/portfolio/rag-evaluation-jp/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


collections on this Qdrant: ['eval_corpus_cl_nagoya_ruri_v3_30m', 'eval_corpus_intfloat_multilingual_e5_base', 'eval_corpus_cl_nagoya_ruri_v3_310m', 'eval_corpus_intfloat_multilingual_e5_large', 'eval_corpus']


## 2. Corpus inventory

The corpus is committed at `data/corpus/`. Each file is a Japanese
technical-doc markdown with H2-sectioned content suitable for the
`markdown_h2` chunker.


In [3]:
corpus_dir = ROOT / "data" / "corpus"
docs = sorted(p for p in corpus_dir.glob("*.md") if p.name != "LICENSES.md")
print(f"corpus docs: {len(docs)}")
for d in docs:
    size_kb = d.stat().st_size / 1024
    print(f"  {d.name:60s}  {size_kb:6.1f} KB")


corpus docs: 15
  jsprimer-array.md                                               60.8 KB
  jsprimer-async.md                                               92.6 KB
  jsprimer-class.md                                               76.3 KB
  jsprimer-error-try-catch.md                                     16.6 KB
  jsprimer-function-declaration.md                                30.7 KB
  jsprimer-json.md                                                11.0 KB
  jsprimer-object.md                                              44.1 KB
  jsprimer-string.md                                              73.1 KB
  react-conditional-rendering.md                                  22.4 KB
  react-lifecycle-of-reactive-effects.md                          83.4 KB
  react-managing-state.md                                         27.7 KB
  react-passing-props-to-a-component.md                           30.5 KB
  react-responding-to-events.md                                   27.9 KB
  react-state-a-compon

## 3. Ingest into Qdrant

We bypass the CLI's argparse and call `collect_chunks` / `upsert_chunks`
directly so the notebook stays self-contained. `--reset` semantics are
achieved by `ensure_collection(reset=True)`.

Note: the embedding model (`intfloat/multilingual-e5-base`) is fetched
from HuggingFace on first use — the first run downloads ~440 MB.


In [4]:
from agent_kms.config import load_config, reset_cache
from agent_kms.ingest import collect_chunks, upsert_chunks
from agent_kms.store import ensure_collection

reset_cache()
cfg = load_config()
print("preset:        ", cfg.preset)
print("knowledge_root:", cfg.knowledge_root)
print("collection:    ", cfg.collection)
print("sources:       ", [s.name for s in cfg.sources])

ensure_collection(reset=True)
chunks = collect_chunks(cfg)
print(f"\ncollected {len(chunks)} chunks")

n = upsert_chunks(cfg, chunks)
print(f"upserted  {n} points into '{cfg.collection}'")


preset:         general
knowledge_root: /Users/toshihiro-yamaguchi/dev/portfolio/rag-evaluation-jp/data/corpus
collection:     eval_corpus
sources:        ['jp_tech_docs']

collected 127 chunks


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 15003.08it/s]

upserted  127 points into 'eval_corpus'


## 4. Corpus statistics

In [5]:
import pandas as pd

rows = []
for c in chunks:
    rows.append({
        "source": Path(c.source_file).name,
        "heading": c.heading,
        "chars": len(c.body),
    })
df_corpus = pd.DataFrame(rows)

print(f"chunks: {len(df_corpus)}")
print(f"docs:   {df_corpus['source'].nunique()}")
print(f"\nchunks per doc:")
print(df_corpus.groupby("source").size().sort_values(ascending=False).to_string())
print(f"\nbody-size summary (chars): min={df_corpus['chars'].min()}, median={int(df_corpus['chars'].median())}, max={df_corpus['chars'].max()}")


chunks: 127
docs:   15

chunks per doc:
source
jsprimer-array.md                         15
jsprimer-string.md                        15
jsprimer-class.md                         14
jsprimer-async.md                         13
jsprimer-object.md                        11
jsprimer-function-declaration.md          10
react-managing-state.md                    9
jsprimer-error-try-catch.md                8
react-passing-props-to-a-component.md      7
jsprimer-json.md                           6
react-state-a-components-memory.md         5
react-responding-to-events.md              4
react-your-first-component.md              4
react-conditional-rendering.md             3
react-lifecycle-of-reactive-effects.md     3

body-size summary (chars): min=176, median=1623, max=47097


## Findings

- **Corpus size**: 15 Japanese technical-doc markdown files → ~130 H2 chunks.
- **Sources**: React 日本語ドキュメント (7 files, CC-BY-4.0) +
  JavaScript Primer 日本語版 (8 files, CC-BY-4.0).
- **Heading distribution is uneven** — the largest files (`async`, `string`,
  `array`, `lifecycle-of-reactive-effects`) contribute 13-15 chunks each;
  smaller pages like `conditional-rendering` contribute only 3.
  This is expected for the `markdown_h2` chunker on real-world docs.
- **Median chunk size ≈ a few hundred characters**; well above the
  `min_chars=80` floor.

This collection is the input to every subsequent notebook.
